# 阶段 4：正式训练（4090 服务器 + Docker + 1.5B 模型）

> **目标**：在 4090 (48GB) 上用更大的模型和更大的组做正式 GRPO 训练，观察更明显的效果。

## 和阶段 2 的区别

| | 阶段 2 | 阶段 4 |
|--|--------|--------|
| **GPU** | 5070 Ti (16GB) | 4090 (48GB) |
| **模型** | Qwen2.5-0.5B | Qwen2.5-1.5B |
| **组大小 G** | 6 | 8 |
| **vLLM** | ❌ 不用 | ✅ 加速生成 |
| **Docker** | ❌ Conda | ✅ Docker 隔离 |
| **奖励函数** | 仅正确性 | 正确性 + 格式 |
| **训练步数** | 300 | 500 |

## 你将学到

1. 为什么 G=8 比 G=6 更好（统计学原理）
2. vLLM 如何加速生成
3. Docker 如何隔离环境
4. 多奖励函数如何组合
5. 1.5B 模型和 0.5B 模型的训练效果差异

## 文件结构

```
stage4_train.py     ← 训练脚本（在服务器上运行）
stage4_deploy.sh    ← 部署脚本（自动构建 Docker + 运行）
Dockerfile          ← Docker 镜像定义
stage4_notebook.ipynb ← 你正在看的这个 notebook（教学+分析）
```

## Cell 1：为什么 G=8 比 G=6 更好？

### 从概率论理解

你在概率论学过：**样本量越大，估计越准确**。

GRPO 的优势计算本质是估计组内的均值和标准差：

```
A_i = (r_i - mean) / std
```

这个 `mean` 和 `std` 是从 G 个样本中估计的。根据统计学：

| G | 均值估计标准误差 | 含义 |
|---|-----------------|------|
| 4 | σ/2 | 估计较粗糙 |
| 6 | σ/2.45 | 好一些 |
| 8 | σ/2.83 | 更准确 |
| 16 | σ/4 | 很准确 |

标准误差 ∝ 1/√G，所以 G 越大，均值和标准差估计越准，优势 A 越可靠。

### 从 `frac_reward_zero_std` 理解

你在阶段 2 学到的最重要的指标：`frac_reward_zero_std`（组内全对或全错的比例）。

| G | 组内全对或全错的概率（假设单次正确率 p=0.5） |
|---|-------------------------------------------|
| 4 | 2 × 0.5⁴ = 12.5% |
| 6 | 2 × 0.5⁶ = 3.1% |
| 8 | 2 × 0.5⁸ = 0.8% |

**G 越大，组内全对或全错的概率越低**，有学习信号的步数比例越高。

### 显存代价

G 越大，每步需要生成和计算的序列越多，显存越大：

| G | 每步序列数 (batch×G) | 0.5B 显存 | 1.5B 显存 |
|---|---------------------|----------|----------|
| 4 | 4×4=16 | ~8 GB | ~20 GB |
| 6 | 2×6=12 | ~10 GB | ~24 GB |
| 8 | 4×8=32 | ~14 GB | ~35 GB |

4090 (48GB) 跑 1.5B + G=8 有足够余量。5070 Ti (16GB) 跑不了。

In [ ]:
# ============================================================
# 演示：G 越大，frac_reward_zero_std 越低
# ============================================================
import numpy as np

def simulate_zero_std_prob(p, G, num_trials=100000):
    """模拟在正确率 p 下，组大小 G 的 zero_std 概率"""
    count = 0
    for _ in range(num_trials):
        rewards = np.random.binomial(1, p, size=G)
        if rewards.std() == 0:  # 全对或全错
            count += 1
    return count / num_trials

print("=== frac_reward_zero_std 模拟 ===")
print(f"{'正确率 p':>8} | {'G=4':>8} | {'G=6':>8} | {'G=8':>8} | {'G=16':>8}")
print("-" * 50)
for p in [0.3, 0.5, 0.7]:
    probs = [simulate_zero_std_prob(p, G) for G in [4, 6, 8, 16]]
    print(f"{p:>8.1f} | {probs[0]:>8.1%} | {probs[1]:>8.1%} | {probs[2]:>8.1%} | {probs[3]:>8.1%}")

print()
print("结论：G 越大，zero_std 概率越低，有学习信号的步数越多")
print("      p=0.5 时，G=8 的 zero_std 只有 0.8%，几乎每步都有学习信号")

## Cell 2：vLLM 是什么？为什么能加速？

### 问题：HF generate() 很慢

阶段 2 和 3 都用 `model.generate()` 生成回答。这个方法逐 token 生成，每次只处理一个 token，GPU 利用率低。

### vLLM 的解决方案

vLLM 是一个高性能 LLM 推理引擎，核心创新：

1. **PagedAttention**：像操作系统的虚拟内存一样管理 KV Cache，减少碎片
2. **连续批处理**：不等一批全部生成完，有新的请求随时加入
3. **高效前缀缓存**：同一个 prompt 生成 G 个回答时，prompt 部分只算一次

### 对 GRPO 的意义

GRPO 每步要生成 G×batch 个回答。以阶段 4 为例：

| | HF generate() | vLLM |
|--|---------------|------|
| 每步生成数 | 4×8=32 个 | 32 个 |
| 生成速度 | ~5 token/s/GPU | ~500 token/s/GPU |
| 每步耗时 | ~30s | ~3s |
| 500 步总时间 | ~4.2 小时 | ~25 分钟 |

（以上为粗略估计，实际取决于硬件和模型）

### 在 TRL 中使用 vLLM

```python
config = GRPOConfig(
    use_vllm=True,
    vllm_mode="colocate",  # vLLM 和训练共享同一 GPU
)
```

`colocate` 模式：vLLM 和训练模型在同一张 GPU 上，省去多卡通信开销。

In [ ]:
# ============================================================
# 演示：vLLM vs HF generate 速度对比（概念演示）
# ============================================================

# 这个 Cell 只展示概念，实际对比需要在服务器上跑
# 这里用模拟数据展示速度差异

import time

print("=== vLLM vs HF generate 速度对比（模拟） ===")
print()
print("场景：Qwen2.5-1.5B，每步生成 32 个回答，每个 32 token")
print()

# 模拟数据
methods = {
    "HF generate()": {"tokens_per_sec": 50, "color": "\033[91m"},  # 红
    "vLLM": {"tokens_per_sec": 800, "color": "\033[92m"},  # 绿

}

total_tokens = 32 * 32  # 32 个回答 × 32 token

for name, info in methods.items():
    time_per_step = total_tokens / info["tokens_per_sec"]
    total_time = time_per_step * 500  # 500 步
    print(f"{info['color']}{name:>20}\033[0m: "
          f"{info['tokens_per_sec']:>6} tok/s | "
          f"每步 {time_per_step:.1f}s | "
          f"500 步 {total_time/60:.1f} 分钟")

print()
print("实际速度取决于硬件，但 vLLM 通常快 5-10 倍")
print("在 4090 上，vLLM 加速更明显，因为 4090 的计算能力更强）

## Cell 3：Docker 是什么？为什么用它？

### 类比

你在阶段 0 学过用 Conda 隔离环境。Docker 做的事情类似，但更彻底：

| | Conda | Docker |
|--|--------|--------|
| 隔离级别 | Python 包 | 整个操作系统 |
| 共享什么 | 系统库、驱动 | 只共享内核和 GPU 驱动 |
| 可重现性 | 一般（依赖系统状态） | 完全（镜像一模一样） |
| 适合场景 | 本机开发 | 服务器部署 |

### 为什么阶段 4 要用 Docker

1. **可重现**：Dockerfile 记录了所有依赖，任何人构建都能得到相同环境
2. **隔离**：不污染服务器的 Python 环境
3. **方便迁移**：换服务器只需把 Dockerfile 和代码拷过去

### Dockerfile 解读

```dockerfile
FROM nvidia/cuda:12.4.1-cudnn-devel-ubuntu22.04  # 基础镜像：CUDA 12.4
RUN apt-get install python3.10 ...                # 安装 Python
RUN pip install torch --index-url ...cu124        # 安装 PyTorch
RUN pip install trl vllm ...                      # 安装 TRL + vLLM
WORKDIR /workspace                                # 工作目录
```

### 部署流程

```
1. SSH 连接服务器
2. 把项目文件传到服务器
3. 安装 Docker（如果没有）
4. 运行 stage4_deploy.sh
   → 构建 Docker 镜像（首次 10-15 分钟）
   → 启动容器运行训练
5. 训练完成后查看输出和 TensorBoard
```

In [ ]:
# ============================================================
# 显示 Dockerfile 内容
# ============================================================
with open('Dockerfile') as f:
    print("=== Dockerfile ===")
    print(f.read())

print("\n=== stage4_deploy.sh ===")
with open('stage4_deploy.sh') as f:
    print(f.read())

## Cell 4：训练脚本解读

### 和阶段 2 的关键区别

#### 1. 多奖励函数

阶段 2 只有一个 `correctness_reward`。阶段 4 加了 `format_reward`：

```python
reward_funcs=[correctness_reward, format_reward]
```

- `correctness_reward`：答案正确得 1.0
- `format_reward`：回答是纯数字得 0.2

两个奖励**相加**作为总奖励。这和阶段 3 实验中 transparent-grpo 的 partial reward 思路一致——给模型更细粒度的信号。

#### 2. vLLM 加速

```python
use_vllm=True,
vllm_mode="colocate",
```

#### 3. 更大的 G

```python
num_generations=8,  # 阶段 2 是 6
```

#### 4. 整除约束

和阶段 2 一样，`batch × accum` 必须能被 G 整除：
- `per_device_train_batch_size=4, gradient_accumulation_steps=2`
- `4 × 2 = 8`，能被 `G=8` 整除 ✅

In [ ]:
# ============================================================
# 显示训练脚本
# ============================================================
with open('stage4_train.py') as f:
    print(f.read())

## Cell 5：部署步骤（在服务器上操作）

### 前提条件

- 服务器可以 SSH 连接
- 服务器有 NVIDIA 驱动（`nvidia-smi` 能跑）
- 服务器上 `/data/models/Qwen2.5-1.5B-Instruct` 已存在

### 步骤 1：SSH 连接服务器

```bash
ssh user@server_ip
```

### 步骤 2：传输项目文件

在**本机**执行：

```bash
scp -r "/data/RL/The Illustrated GRPO/" user@server_ip:/data/RL/
```

或者用 rsync（更快，支持断点续传）：

```bash
rsync -avz --progress "/data/RL/The Illustrated GRPO/" user@server_ip:/data/RL/The\ Illustrated\ GRPO/
```

### 步骤 3：安装 Docker（如果服务器没有）

```bash
# 安装 Docker
curl -fsSL https://get.docker.com | sh
sudo usermod -aG docker $USER
sudo systemctl start docker

# 安装 NVIDIA Container Toolkit（让 Docker 能用 GPU）\ndistribution=$(. /etc/os-release;echo $ID$VERSION_ID)
curl -s -L https://nvidia.github.io/libnvidia-container/gpgkey | sudo apt-key add -
curl -s -L https://nvidia.github.io/libnvidia-container/$distribution/libnvidia-container.list | sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list
sudo apt-get update && sudo apt-get install -y nvidia-container-toolkit
sudo systemctl restart docker
```

### 步骤 4：运行部署脚本

```bash
cd "/data/RL/The Illustrated GRPO"
bash stage4_deploy.sh
```

脚本会自动：
1. 检查 Docker 和 GPU 环境
2. 构建 Docker 镜像（首次约 10-15 分钟）
3. 启动容器运行训练

### 步骤 5：查看训练进度

训练日志会实时输出到终端。如果想后台运行：

```bash
nohup bash stage4_deploy.sh > train.log 2>&1 &
tail -f train.log  # 查看进度
```

### 步骤 6：查看 TensorBoard

```bash
# 在服务器上启动 TensorBoard
tensorboard --logdir=logs/stage4 --host=0.0.0.0 --port=6006

# 在本机建立 SSH 隧道
ssh -L 6006:localhost:6006 user@server_ip

# 然后在本机浏览器打开 http://localhost:6006
```

In [ ]:
# ============================================================
# 检查本机是否有 1.5B 模型（用于验证路径）
# ============================================================
import os

model_path = "/data/models/Qwen2.5-1.5B-Instruct"
if os.path.exists(model_path):
    files = os.listdir(model_path)
    print(f"✅ 1.5B 模型存在: {model_path}")
    print(f"   文件: {', '.join(files[:5])}...")
else:
    print(f"❌ 1.5B 模型不存在: {model_path}")
    print("   需要先下载：")
    print("   from modelscope import snapshot_download")
    print(f"   snapshot_download('Qwen/Qwen2.5-1.5B-Instruct', cache_dir='/data/models')")

# 检查项目文件
print()
for f in ['stage4_train.py', 'stage4_deploy.sh', 'Dockerfile']:
    path = os.path.join('.', f)
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"✅ {f} ({size} bytes)")
    else:
        print(f"❌ {f} 不存在")

## Cell 6：训练后分析

训练完成后，在服务器上运行以下命令查看结果：

### TensorBoard 指标关注点

| 指标 | 期望趋势 | 和阶段 2 对比 |
|------|---------|-------------|
| `reward` | 上升 | 应该比阶段 2 更稳定（G=8 > G=6）|
| `frac_reward_zero_std` | 下降 | 应该比阶段 2 更低（G=8 的 zero_std 概率更低）|
| `kl` | 保持在 2 以下 | 1.5B 模型可能更稳定 |
| `grad_norm` | 不爆炸 | 4090 显存大，batch 更大，梯度更平滑 |
| `entropy` | 缓慢下降 | 1.5B 模型初始更确定 |

### 0.5B vs 1.5B 预期差异

| | 0.5B (阶段 2) | 1.5B (阶段 4) |
|--|---------------|---------------|
| 基座准确率 | ~50% | ~70-80% |
| 训练后准确率 | ~80% | ~90%+ |
| 训练稳定性 | 一般 | 更好（模型更大，梯度更平滑）|
| 收敛速度 | 较慢 | 较快（模型容量大）|

### 注意

如果 1.5B 基座准确率已经很高（>80%），`frac_reward_zero_std` 可能反而高——因为组内全对的概率大了。这时可能需要用更难的题（四位数加法）来保持组内多样性。

In [ ]:
# ============================================================
# 训练后对比测试脚本（在服务器上运行）
# ============================================================
# 这个脚本可以在训练完成后运行，对比训练前后效果
# 保存为 stage4_eval.py 在服务器上运行

eval_script = '''"""
阶段 4 评估：对比 1.5B 模型训练前后准确率
"""
import torch
import re
import random
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = "/data/models/Qwen2.5-1.5B-Instruct"
TRAINED_PATH = "output/grpo_1.5b_addition"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# 生成测试题
random.seed(999)
test_questions = []
for _ in range(20):
    a = random.randint(100, 999)
    b = random.randint(100, 999)
    test_questions.append((a, b, str(a + b)))

def test_model(model, tokenizer, questions):
    correct = 0
    for a, b, ans in questions:
        messages = [{"role": "user", "content": f"What is {a}+{b}? Answer with just the number."}]
        input_ids = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=True, return_dict=False
        ).to(model.device)
        with torch.no_grad():
            output = model.generate(input_ids, max_new_tokens=16, temperature=0.0, do_sample=False)
        response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        numbers = re.findall(r'\d+', response)
        if numbers and numbers[-1] == ans:
            correct += 1
    return correct / len(questions)

# 测试训练前
print("=== 测试训练前 ===")
base_model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16, device_map="auto")
base_acc = test_model(base_model, tokenizer, test_questions)
print(f"准确率: {base_acc:.1%}")

# 测试训练后
print("\\n=== 测试训练后 ===")
trained_model = AutoModelForCausalLM.from_pretrained(TRAINED_PATH, torch_dtype=torch.bfloat16, device_map="auto")
trained_acc = test_model(trained_model, tokenizer, test_questions)
print(f"准确率: {trained_acc:.1%}")

print(f"\\n=== 对比 ===")
print(f"训练前: {base_acc:.1%}")
print(f"训练后: {trained_acc:.1%}")
print(f"提升: {trained_acc - base_acc:+.1%}")

del base_model, trained_model
torch.cuda.empty_cache()
'''

print(eval_script)
print("\n--- 将以上脚本保存为 stage4_eval.py，在服务器上训练完成后运行 ---")

## Cell 7：理解检查点

1. **为什么 G=8 比 G=6 更好？**
   - 提示：样本量越大，均值/标准差估计越准；G 越大，zero_std 概率越低

2. **vLLM 为什么比 HF generate() 快？**
   - 提示：PagedAttention + 连续批处理 + 前缀缓存

3. **Docker 相比 Conda 有什么优势？**
   - 提示：隔离更彻底、可重现性更好、适合服务器部署

4. **阶段 4 为什么加了 format_reward？**
   - 提示：给模型更细粒度的信号，增加组内奖励多样性

5. **如果 1.5B 基座准确率已经 80%，应该怎么调整？**
   - 提示：用更难的题（四位数加法）保持组内多样性

### 下一步

1. 在本机确认所有文件正确（Cell 5 检查）
2. SSH 到服务器，传输文件
3. 安装 Docker（如果需要）
4. 运行 `bash stage4_deploy.sh`
5. 训练完成后运行评估脚本
6. 对比 0.5B（阶段 2）和 1.5B（阶段 4）的训练效果